In [ ]:
# Remove all the commas in the end of the line (Known issue in some of the files in the weather dataset)
#! perl -i -pe 's/,(?=\s*$)|,$//' *.csv

#Covert the CSV file's Years into four digits addressing the Y2K problem (Known issue in some of the weather dataset)
#perl -pe 's{^(\d{1,2}/\d{1,2}/)(\d{2})\b}{$1 . ($2<=68?2000+$2:1900+$2)}e' weatherdata-682-1478.csv > output.csv

In [ ]:
"""
df_wildfire
    │
    ├── find_nearest_weather_coordinate()   ← already built
    │        assigns NEAREST_WEATHER_LAT/LON to each fire
    │
    ├── for each fire record:
    │        look up weather_df where STATION matches nearest
    │        and DATE is within the pre-fire window
    │        → aggregate into fixed feature columns
    │
    └── result: training dataset:
                one row per wildfire event
                with weather features as columns
                + your WEATHER_FIRE_RISK_SCORE as target (or label)
"""

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import math
import folium
from IPython.display import display, IFrame
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Show all rows and columns of a Pandas datafra,e in Jupyter notebook (It may hang the page if the dataframe is big)

# Show all rows
#pd.set_option('display.max_rows', None)

# Show all columns
#pd.set_option('display.max_columns', None)

In [ ]:
"""
Target location: Fairbanks and max_radius to consider
"""
TARGET_LAT = 64.8378 #north
TARGET_LON = -147.7164 #West
MAX_RADIUS = 250 #miles

In [ ]:
def readAllCSVFiles(folder_path):
    """
    Read all CSV files inside a folder into a Pandas Dataframe.
    Input: folder path
    Output: pandas dataframe
    """
    # 1. Define the path to your folder
    folder_path = folder_path + "/*.csv"
    # 2. Get list of files, read them, and concatenate vertically
    all_files = glob.glob(folder_path)
    
    df = pd.concat((pd.read_csv(file) for file in all_files), ignore_index=False)
    
    return df

In [ ]:
def calculate_haversine_distance(lat1, lon1, lat2, lon2, unit='km'):
    """
    Calculate the great-circle distance between two points on Earth.
    Input: coordinates (latitude and longitude) of two locations
    Output: distance between those two points
    """
    # Earth radius in kilometers (use 3958.8 for miles)
    R = 6371.0 

    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    distance = R * c

    if unit == 'miles':
        return distance * 0.621371
    return distance # defaults to kilometers

In [ ]:
def find_nearest_weather_coordinate(df_wildfire, unique_weather_coordinates):
    """
    For each record in df_wildfire, find the nearest coordinate pair from
    unique_weather_coordinates using the Haversine great-circle distance.

    A cross join is performed between df_wildfire and unique_weather_coordinates
    to produce every possible wildfire-to-weather coordinate pair. The Haversine
    distance is then computed for each pair, and only the row with the minimum
    distance is kept for each wildfire record.

    Parameters
    ----------
    df_wildfire : pd.DataFrame
        DataFrame of wildfire records. Must contain the columns:
        - 'LATITUDE'  : float, latitude of the wildfire location.
        - 'LONGITUDE' : float, longitude of the wildfire location.

    unique_weather_coordinates : pd.DataFrame
        DataFrame of candidate weather station coordinates. Must contain:
        - 'Latitude'  : float, latitude of the weather coordinate.
        - 'Longitude' : float, longitude of the weather coordinate.

    Returns
    -------
    pd.DataFrame
        A copy of df_wildfire with three new columns appended:
        - 'NEAREST_WEATHER_LAT'  : float, latitude of the closest weather coordinate.
        - 'NEAREST_WEATHER_LON'  : float, longitude of the closest weather coordinate.
        - 'NEAREST_WEATHER_DIST' : float, Haversine distance (in whatever unit
                                   calculate_haversine_distance returns) to that
                                   closest coordinate.
    """
    # Cross join: every wildfire record paired with every weather coordinate
    crossed = df_wildfire.merge(unique_weather_coordinates, how='cross')

    # Compute Haversine distance for every pair in one vectorised apply
    crossed['NEAREST_WEATHER_DIST'] = crossed.apply(
        lambda row: calculate_haversine_distance(
            row['LATITUDE'], row['LONGITUDE'],
            row['Latitude'], row['Longitude'], unit = 'miles'
        ), axis=1
    )
    # Keep only the closest weather coordinate for each wildfire record
    crossed = crossed.loc[crossed.groupby('ID')['NEAREST_WEATHER_DIST'].idxmin()]

    # Rename matched columns and attach to a clean copy of df_wildfire
    df = df_wildfire.copy()
    df['NEAREST_WEATHER_LAT']  = crossed['Latitude'].values
    df['NEAREST_WEATHER_LON']  = crossed['Longitude'].values
    df['NEAREST_WEATHER_DIST'] = crossed['NEAREST_WEATHER_DIST'].values

    return df

In [ ]:
def filter_data_within_max_radius(df, target_lat, target_lon, current_lat, current_lon, max_radius, unit='miles'):
    """
    Filter a DataFrame down to only the records that fall within a given
    radius (in miles) of a target coordinate, using the Haversine formula
    to calculate great-circle distance between each record and the target.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing location records with latitude/longitude columns.
    target_lat : float
        Latitude of the reference point to measure distance from.
    target_lon : float
        Longitude of the reference point to measure distance from.
    current_lat : float
        Name of the column in `df` holding each record's latitude.
    current_lon : float
        Name of the column in `df` holding each record's longitude.
    max_radius : float
        Maximum allowed distance, in miles, from the target coordinate.
        Records farther than this are excluded.

    Returns
    -------
    pd.DataFrame
        A filtered copy of `df` containing only records within `max_radius`
        miles of the target coordinate, with an added `distance_miles`
        column showing each record's computed distance.
    """
    
    df['distance_miles'] = df.apply(
    lambda row: calculate_haversine_distance(target_lat, target_lon, row[current_lat], row[current_lon], unit), axis=1
    )
    # Filter for rows within max_radius
    filtered_df = df[df['distance_miles'] <= max_radius]
    return filtered_df

In [ ]:
def calculate_fire_weather_risk(df_wildfire):
    """
    Take a DataFrame of AK fire location records and append weather-relevant
    fire risk sub-scores plus a composite 0-100 score.
 
    Parameters
    ----------
    df_wildfire : pd.DataFrame
        DataFrame already loaded from the fire location CSV
        (e.g. AK_fire_location_points_NAD83.csv).
 
    Returns
    -------
    pd.DataFrame
        The original records with these columns appended:
        IGNITION_WEATHER_SCORE, SEASONAL_SCORE, ASPECT_SCORE, ELEVATION_SCORE,
        SLOPE_WIND_SCORE, SPREAD_RATE_SCORE, WEATHER_FIRE_RISK_SCORE,
        WEATHER_FIRE_RISK_CATEGORY, DISCOVERY_MONTH, ACRES_PER_DAY
    """
    """
    WEATHER_FIRE_RISK_SCORE (0–100) components:
    - Ignition–weather link (0–15): lightning ignitions score highest since 
      they're a direct product of convective storm activity; human-caused fires 
      score low since weather only plays a secondary (fuel-drying) role in those
    - Seasonal fire-weather climatology (0–20): scored by discovery month against 
      Alaska's known fire-weather season, peaking June–July when temperatures, 
      dryness, and instability are highest
    - Solar aspect exposure (0–15): south/southwest-facing origins score higher 
      since they receive more solar loading and dry out faster under the same regional conditions
    - Elevation/fuel-moisture band (0–15): mid-elevation zones (1,500–2,500 ft) 
      score highest, reflecting Alaska's interior black-spruce fuel belt; 
      wet lowlands and sparse alpine zones score lower
    - Slope–wind interaction (0–10): steeper terrain amplifies wind-driven fire behavior
    - Fire spread rate (0–25): acres burned per day between discovery and 
      control/out date — this is the strongest empirical fingerprint of severe 
      fire weather (low humidity, high wind, atmospheric instability) actually acting 
      on a fire after ignition; falls back to a smaller size-only score when no resolvable duration exists

    Dropped suppression strategy and incident management complexity from the v1 score since those reflect firefighting decisions, not weather. Distribution shifted toward the middle (mean ~45 vs ~20 before) since most fires now register at least moderate seasonal/aspect exposure, with the spread-rate component doing the heavy lifting in separating high-risk outliers — June fires with explosive day-over-day growth dominate the top of the list, which lines up with how fire-weather-driven events actually present.
    """
    df = df_wildfire
 
    # Parse relevant dates
    disc = pd.to_datetime(df['DISCOVERYDATETIME'], errors='coerce')
    out = pd.to_datetime(df['OUTDATE'], errors='coerce')
    ctrl = pd.to_datetime(df['CONTROLDATETIME'], errors='coerce')
    end_date = out.fillna(ctrl)
 
    # --- 1. IGNITION-WEATHER LINK SCORE (0-15) ---
    def ignition_score(cause):
        if pd.isna(cause):
            return 5
        c = str(cause).strip().lower()
        if 'lightning' in c:
            return 15
        if c in ('natural', 'natural out'):
            return 12
        if 'undetermined' in c or 'investigat' in c or c in ('unknown', ''):
            return 6
        if 'prescribed' in c:
            return 0
        return 4  # human-caused fires: weather only a secondary drying factor
    ignition_weather_score = df['GENERALCAUSE'].apply(ignition_score)
 
    # --- 2. SEASONAL FIRE-WEATHER CLIMATOLOGY SCORE (0-20) ---
    month_score = {
        1: 1, 2: 1, 3: 3, 4: 9, 5: 17, 6: 20, 7: 20,
        8: 15, 9: 7, 10: 3, 11: 1, 12: 1
    }
    seasonal_score = disc.dt.month.map(month_score).fillna(8)
 
    # --- 3. SOLAR ASPECT EXPOSURE SCORE (0-15) ---
    aspect_map = {
        'south': 15, 's': 15, 'south west': 14, 'sw': 14, 'south east': 13, 'se': 13,
        'west': 10, 'w': 10, 'east': 9, 'e': 9,
        'north west': 5, 'nw': 5, 'north east': 4, 'ne': 4, 'north': 3, 'n': 3,
        'flat': 8, 'unknown': 7,
    }
    aspect_score = df['ORIGINASPECT'].astype(str).str.strip().str.lower().map(aspect_map).fillna(7)
 
    # --- 4. ELEVATION / FUEL-MOISTURE BAND SCORE (0-15) ---
    elev_map = {
        '0-500': 5, '501-1500': 10, '0501-1500': 10, '1501-2500': 15, '2501-3500': 12,
        '3501-4500': 8, '4501-5500': 5, '5501-6500': 3, '6501-7500': 2,
        '7501-8500': 1, '8501+': 1, 'unknown': 7,
    }
    elevation_score = df['ORIGINELEVATION'].astype(str).str.strip().str.lower().map(elev_map).fillna(7)
 
    # --- 5. SLOPE-WIND INTERACTION SCORE (0-10) ---
    slope_map = {'0-25': 2, '26-40': 4, '41-55': 6, '56-75': 8, '76+': 10, 'Unknown': 3}
    slope_score = df['ORIGINSLOPE'].astype(str).str.strip().map(slope_map).fillna(3)
 
    # --- 6. FIRE SPREAD RATE SCORE (0-25) ---
    acres = pd.to_numeric(df['ESTIMATEDTOTALACRES'], errors='coerce').fillna(0).clip(lower=0)
    duration_days = (end_date - disc).dt.total_seconds() / 86400
    duration_days = duration_days.where(duration_days >= 1, 1)
    spread_rate = acres / duration_days
    has_duration = duration_days.notna() & end_date.notna()
 
    log_spread = np.log1p(spread_rate.where(has_duration, np.nan))
    log_acres_fallback = np.log1p(acres)
 
    max_log_spread = log_spread.max()
    spread_score = (log_spread / max_log_spread) * 25 if max_log_spread > 0 else log_spread * 0
 
    max_log_acres = log_acres_fallback.max()
    fallback_score = (log_acres_fallback / max_log_acres) * 15 if max_log_acres > 0 else log_acres_fallback * 0
    spread_score = spread_score.fillna(fallback_score).fillna(0)
 
    # --- COMPOSITE WEATHER-RELEVANT FIRE RISK SCORE (0-100) ---
    weather_fire_risk_score = (
        ignition_weather_score + seasonal_score + aspect_score +
        elevation_score + slope_score + spread_score
    )
 
    # Append everything to the same DataFrame
    df['DISCOVERY_MONTH'] = disc.dt.month
    df['ACRES_PER_DAY'] = spread_rate.round(3)
    df['IGNITION_WEATHER_SCORE'] = ignition_weather_score.round(2)
    df['SEASONAL_SCORE'] = seasonal_score.round(2)
    df['ASPECT_SCORE'] = aspect_score.round(2)
    df['ELEVATION_SCORE'] = elevation_score.round(2)
    df['SLOPE_WIND_SCORE'] = slope_score.round(2)
    df['SPREAD_RATE_SCORE'] = spread_score.round(2)
    df['WEATHER_FIRE_RISK_SCORE'] = weather_fire_risk_score.round(2)
    df['WEATHER_FIRE_RISK_CATEGORY'] = pd.cut(
        df['WEATHER_FIRE_RISK_SCORE'],
        bins=[-0.1, 20, 40, 60, 80, 100],
        labels=['Very Low', 'Low', 'Moderate', 'High', 'Extreme']
    )
 
    return df

In [ ]:
# Extract weather features (Multiple funtions are written in the same cell)

# =============================================================================
# HELPER — Slice the pre-fire weather window for one fire record
# =============================================================================

def _get_weather_window(df_weather, station_lat, station_lon, discovery_date, lookback_days):
    """
    Return the weather rows for a specific station in the N days strictly
    before discovery_date (DATE < discovery_date), preventing data leakage.

    Parameters
    ----------
    df_weather : pd.DataFrame
        Weather time-series with columns: Latitude, Longitude, Date,
        Max Temperature, Min Temperature, Precipitation, Wind,
        Relative Humidity, Solar.
    station_lat : float
        Latitude of the nearest weather station for this fire.
    station_lon : float
        Longitude of the nearest weather station for this fire.
    discovery_date : pd.Timestamp
        Fire discovery date/time.
    lookback_days : int
        Number of days before discovery_date to include.

    Returns
    -------
    pd.DataFrame
        Filtered slice of df_weather for that station and time window.
    """
    cutoff = discovery_date - pd.Timedelta(days=lookback_days)
    mask = (
        (df_weather['Latitude']  == station_lat) &
        (df_weather['Longitude'] == station_lon) &
        (df_weather['Date']      >= cutoff) &
        (df_weather['Date']      <  discovery_date)
    )
    return df_weather.loc[mask].copy()


# =============================================================================
# FEATURE FUNCTION 1 — Short-term drying (last 7 days)
# =============================================================================

def compute_short_term_drying_features(df_weather, station_lat, station_lon, discovery_date):
    """
    Compute short-term (7-day pre-fire) atmospheric drying features for a
    single fire event. These capture the immediate weather conditions that
    drive ignition risk and rapid early spread.

    Parameters
    ----------
    df_weather : pd.DataFrame
        Weather time-series (see load_weather_data for column details).
    station_lat : float
        Latitude of the nearest weather station.
    station_lon : float
        Longitude of the nearest weather station.
    discovery_date : pd.Timestamp
        Fire discovery date/time.

    Returns
    -------
    dict
        avg_max_temp_7d    : mean daily max temperature over 7 days (°C).
        avg_min_temp_7d    : mean daily min temperature over 7 days (°C).
        avg_rh_7d          : mean relative humidity over 7 days (0–1).
        total_precip_7d    : total precipitation over 7 days (mm).
        max_wind_7d        : highest daily average wind speed over 7 days (m/s).
        consec_dry_days_7d : count of days with zero precipitation.
        avg_solar_7d       : mean daily solar radiation over 7 days (W/m²).
    """
    w = _get_weather_window(df_weather, station_lat, station_lon, discovery_date, lookback_days=7)

    return {
        'avg_max_temp_7d':    w['Max Temperature'].mean(),
        'avg_min_temp_7d':    w['Min Temperature'].mean(),
        'avg_rh_7d':          w['Relative Humidity'].mean(),
        'total_precip_7d':    w['Precipitation'].sum(),
        'max_wind_7d':        w['Wind'].max(),
        'consec_dry_days_7d': int((w['Precipitation'] == 0).sum()),
        'avg_solar_7d':       w['Solar'].mean(),
    }


# =============================================================================
# FEATURE FUNCTION 2 — Medium-term drought signal (last 30 days)
# =============================================================================

def compute_medium_term_drought_features(df_weather, station_lat, station_lon, discovery_date):
    """
    Compute medium-term (30-day pre-fire) drought and fuel-moisture features
    for a single fire event. These reflect cumulative drying that raises fuel
    availability even when the immediate forecast looks mild.

    Parameters
    ----------
    df_weather : pd.DataFrame
        Weather time-series (see load_weather_data for column details).
    station_lat : float
        Latitude of the nearest weather station.
    station_lon : float
        Longitude of the nearest weather station.
    discovery_date : pd.Timestamp
        Fire discovery date/time.

    Returns
    -------
    dict
        avg_max_temp_30d   : mean daily max temperature over 30 days (°C).
        total_precip_30d   : total precipitation over 30 days (mm).
        precip_anomaly_30d : observed precip minus climatological daily mean × 30
                             (negative = drier than average).
        avg_wind_30d       : mean daily wind speed over 30 days (m/s).
        avg_rh_30d         : mean relative humidity over 30 days (0–1).
        days_no_rain_30d   : count of days with zero precipitation.
        avg_solar_30d      : mean daily solar radiation over 30 days (W/m²).
    """
    w = _get_weather_window(df_weather, station_lat, station_lon, discovery_date, lookback_days=30)

    total_precip = w['Precipitation'].sum()
    daily_mean   = w['Precipitation'].mean()
    anomaly      = total_precip - (daily_mean * 30) if not np.isnan(daily_mean) else np.nan

    return {
        'avg_max_temp_30d':   w['Max Temperature'].mean(),
        'total_precip_30d':   total_precip,
        'precip_anomaly_30d': anomaly,
        'avg_wind_30d':       w['Wind'].mean(),
        'avg_rh_30d':         w['Relative Humidity'].mean(),
        'days_no_rain_30d':   int((w['Precipitation'] == 0).sum()),
        'avg_solar_30d':      w['Solar'].mean(),
    }


# =============================================================================
# FEATURE FUNCTION 3 — Seasonal drought build-up (last 90 days)
# =============================================================================

def compute_seasonal_drought_features(df_weather, station_lat, station_lon, discovery_date):
    """
    Compute seasonal (90-day pre-fire) drought accumulation features for a
    single fire event. These capture the slow build-up of fuel dryness over
    the fire season — a key predictor for extreme fire years.

    Parameters
    ----------
    df_weather : pd.DataFrame
        Weather time-series (see load_weather_data for column details).
    station_lat : float
        Latitude of the nearest weather station.
    station_lon : float
        Longitude of the nearest weather station.
    discovery_date : pd.Timestamp
        Fire discovery date/time.

    Returns
    -------
    dict
        avg_max_temp_90d  : mean daily max temperature over 90 days (°C).
        max_temp_90d      : highest recorded daily max temp over 90 days (°C).
        total_precip_90d  : total precipitation over 90 days (mm).
        days_no_rain_90d  : count of days with zero precipitation over 90 days.
        avg_rh_90d        : mean relative humidity over 90 days (0–1).
        total_solar_90d   : cumulative solar radiation over 90 days (W/m²).
    """
    w = _get_weather_window(df_weather, station_lat, station_lon, discovery_date, lookback_days=90)

    return {
        'avg_max_temp_90d':  w['Max Temperature'].mean(),
        'max_temp_90d':      w['Max Temperature'].max(),
        'total_precip_90d':  w['Precipitation'].sum(),
        'days_no_rain_90d':  int((w['Precipitation'] == 0).sum()),
        'avg_rh_90d':        w['Relative Humidity'].mean(),
        'total_solar_90d':   w['Solar'].sum(),
    }


# =============================================================================
# FEATURE FUNCTION 4 — Extreme weather events (last 7 days)
# =============================================================================

def compute_extreme_weather_features(df_weather, station_lat, station_lon, discovery_date,
                                     heat_threshold_c=25.0, wind_threshold_ms=8.0,
                                     low_rh_threshold=0.25):
    """
    Count extreme weather events in the 7 days before fire discovery.
    Extreme heat, high winds, and low humidity are the three atmospheric
    conditions most directly linked to rapid fire growth.

    Thresholds are calibrated to Alaska's climate:
        heat_threshold_c  : 25°C (~77°F) — hot for Alaska, rare but fire-critical.
        wind_threshold_ms : 8 m/s (~18 mph) — sustained wind that drives spread.
        low_rh_threshold  : 0.25 (25%) — critically dry air for fuel ignitability.

    Parameters
    ----------
    df_weather : pd.DataFrame
        Weather time-series (see load_weather_data for column details).
    station_lat : float
        Latitude of the nearest weather station.
    station_lon : float
        Longitude of the nearest weather station.
    discovery_date : pd.Timestamp
        Fire discovery date/time.
    heat_threshold_c : float, optional
        Daily max temperature (°C) above which a day counts as a heat event.
        Default 25°C.
    wind_threshold_ms : float, optional
        Daily average wind speed (m/s) above which a day counts as high-wind.
        Default 8.0 m/s.
    low_rh_threshold : float, optional
        Relative humidity (0–1) below which a day counts as critically dry.
        Default 0.25.

    Returns
    -------
    dict
        days_above_heat_threshold : count of days exceeding heat_threshold_c.
        days_above_wind_threshold : count of days exceeding wind_threshold_ms.
        days_low_rh               : count of days below low_rh_threshold.
        max_temp_7d               : highest daily max temperature in window (°C).
        max_wind_7d               : highest wind speed recorded in window (m/s).
        min_rh_7d                 : lowest relative humidity recorded in window (0–1).
        max_solar_7d              : highest solar radiation recorded in window (W/m²).
    """
    w = _get_weather_window(df_weather, station_lat, station_lon, discovery_date, lookback_days=7)

    return {
        'days_above_heat_threshold': int((w['Max Temperature']   > heat_threshold_c).sum()),
        'days_above_wind_threshold': int((w['Wind']              > wind_threshold_ms).sum()),
        'days_low_rh':               int((w['Relative Humidity'] < low_rh_threshold).sum()),
        'max_temp_7d':               w['Max Temperature'].max(),
        'max_wind_7d':               w['Wind'].max(),
        'min_rh_7d':                 w['Relative Humidity'].min(),
        'max_solar_7d':              w['Solar'].max(),
    }


# =============================================================================
# STEP 4 — Assemble all features for every fire record
# =============================================================================

def build_weather_features(df_wildfire, df_weather):
    """
    Iterate over every fire record in df_wildfire and apply all four feature
    functions using the pre-assigned nearest weather station coordinates and
    the fire's DISCOVERYDATETIME as the reference point.

    All weather features are derived strictly from dates BEFORE
    DISCOVERYDATETIME to prevent data leakage into a prediction model.

    Parameters
    ----------
    df_wildfire : pd.DataFrame
        Wildfire records. Must contain:
        - 'DISCOVERYDATETIME'   : datetime
        - 'NEAREST_WEATHER_LAT' : float
        - 'NEAREST_WEATHER_LON' : float
    df_weather : pd.DataFrame
        Weather time-series loaded via load_weather_data(). Must contain:
        Date, Latitude, Longitude, Max Temperature, Min Temperature,
        Precipitation, Wind, Relative Humidity, Solar.

    Returns
    -------
    pd.DataFrame
        df_wildfire with all computed weather feature columns appended.
    """
    feature_rows = []

    for _, row in df_wildfire.iterrows():
        lat            = row['NEAREST_WEATHER_LAT']
        lon            = row['NEAREST_WEATHER_LON']
        discovery_date = row['DISCOVERYDATETIME']

        features = {}
        features.update(compute_short_term_drying_features(df_weather, lat, lon, discovery_date))
        features.update(compute_medium_term_drought_features(df_weather, lat, lon, discovery_date))
        features.update(compute_seasonal_drought_features(df_weather, lat, lon, discovery_date))
        features.update(compute_extreme_weather_features(df_weather, lat, lon, discovery_date))

        feature_rows.append(features)

    feature_df = pd.DataFrame(feature_rows, index=df_wildfire.index)
    return pd.concat([df_wildfire, feature_df], axis=1)

In [ ]:
def create_map(df, lat_col='LATITUDE', lon_col='LONGITUDE', zoom_start=1,
               center_lat=TARGET_LAT, center_lon=TARGET_LON,
                save_path=None, show_in_notebook=True):
    """
    Create an interactive Folium map with red circle markers for each row
    in the dataframe. Hovering over a circle shows a tooltip with all
    column headers and their corresponding values for that record.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least LATITUDE and LONGITUDE columns.
        lat_col (str): Name of the latitude column.
        lon_col (str): Name of the longitude column.
        zoom_start (int): Initial zoom level of the map.
        save_path (str or None): If provided, saves the map to this HTML file path.
        show_in_notebook (bool): If True, displays the map inline in a Jupyter notebook.

    Returns:
        folium.Map: The generated map object.
    """
    # Drop rows with missing coordinates
    df = df.dropna(subset=[lat_col, lon_col])

    # Center the map on the mean lat/lon
    center_lat = df[lat_col].mean()
    center_lon = df[lon_col].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    for _, row in df.iterrows():
        # Build an HTML table of all column headers and values for this row
        rows_html = "".join(
            f"<tr><td style='padding:2px 6px;font-weight:bold;'>{col}</td>"
            f"<td style='padding:2px 6px;'>{row[col]}</td></tr>"
            for col in df.columns
        )
        tooltip_html = f"<table>{rows_html}</table>"

        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=2,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            tooltip=folium.Tooltip(tooltip_html, sticky=True)
        ).add_to(m)

    # Optionally save to an HTML file
    if save_path:
        m.save(save_path)

    # Optionally display inline in a Jupyter notebook
    if show_in_notebook:
        display(m)

    return m

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_correlation_heatmap(df, cols_to_exclude=None, show_numbers=True):
    """
    Excludes specified columns from a DataFrame, computes the correlation matrix,
    and returns the heatmap plot as a Matplotlib axes object.
    
    Parameters:
    - df: The input pandas DataFrame.
    - cols_to_exclude: List of column names to drop before plotting.
    - show_numbers: Boolean flag to show (True) or hide (False) values inside cells.
    """
    if cols_to_exclude:
        # Exclude the specified columns
        df_subset = df.drop(columns=cols_to_exclude)
    else:
        df_subset = df.copy()
        
    # Calculate the correlation matrix
    corr_matrix = df_subset.corr()
    
    # Draw the heatmap
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(
        corr_matrix, 
        annot=show_numbers,  # Controls display of numbers
        cmap='coolwarm', 
        fmt='.2f', 
        square=True
    )
    
    return ax

In [ ]:
# Read all weather and wildfire data to pandas dataframes and convert date/time columns to_datetime() for timeseries analysis 

df_weather = readAllCSVFiles("../Dataset/WeatherDataFromAcrossAlaska")
#len(df_weather.columns)
#df_weather

df_wildfire = readAllCSVFiles("../Dataset/WildFireDataFromAcrossAlaska")
#len(df_wildfire.columns)
#df_wildfire

In [ ]:
# Get a cumulative weather-aligned risk score for each record in the historic fire database (df_wildfire)

df_wildfire_RiskScore = calculate_fire_weather_risk(df_wildfire)
#len(df_wildfire.columns)
#df_wildfire

In [ ]:
# Get all the unique weather-station-coordinate (Latitude, Longitude)

unique_weather_coordinates = df_weather[['Longitude', 'Latitude']].drop_duplicates()
#len(unique_weather_coordinates)
#unique_weather_coordinates

In [ ]:
# Map the nearest weather station coordinate (from unique_weather_coordinates) for each records in the historic fire database (df_wildfire)

df_combined_wildfire_RiskScore_NearesWeatherStationCoordinate = find_nearest_weather_coordinate(df_wildfire, unique_weather_coordinates)
#df_combined_wildfire_RiskScore_NearesWeatherStationCoordinate.columns

In [ ]:
#Convert the data to_datetime for timeseries analysis

df_weather['Date'] = pd.to_datetime(df_weather['Date'], errors='coerce')
df_combined_wildfire_RiskScore_NearesWeatherStationCoordinate['DISCOVERYDATETIME'] = pd.to_datetime(df_combined_wildfire_RiskScore_NearesWeatherStationCoordinate['DISCOVERYDATETIME'], errors='coerce')

In [ ]:
#Preparing the Training Data Super Set with all the columns
training_data = build_weather_features(df_combined_wildfire_RiskScore_NearesWeatherStationCoordinate, df_weather)

In [ ]:
#Get the exact data set for training
filtered_training_data = training_data[training_data['DISCOVERYDATETIME'].between('1990-05-01', '2012-08-30')]
filtered_training_data_selectedColumn = filtered_training_data[['LATITUDE', 'LONGITUDE', 'NEAREST_WEATHER_LAT', 'NEAREST_WEATHER_LON', 'DISCOVERYDATETIME',  
                                                                  'avg_max_temp_7d', 'avg_min_temp_7d', 'avg_rh_7d', 'total_precip_7d', 'max_wind_7d', 'consec_dry_days_7d', 'avg_solar_7d', 
                                                                  'max_temp_7d', 'min_rh_7d', 'max_solar_7d',
                                                                  'avg_max_temp_30d', 'total_precip_30d', 'precip_anomaly_30d', 'avg_wind_30d', 'avg_rh_30d', 'days_no_rain_30d', 'avg_solar_30d', 
                                                                  'avg_max_temp_90d', 'max_temp_90d', 'total_precip_90d', 'days_no_rain_90d', 'avg_rh_90d', 'total_solar_90d', 
                                                                  'days_above_heat_threshold', 'days_above_wind_threshold', 'days_low_rh', 
                                                                  'IGNITION_WEATHER_SCORE', 'SEASONAL_SCORE', 'ASPECT_SCORE', 'ELEVATION_SCORE', 'SLOPE_WIND_SCORE', 'SPREAD_RATE_SCORE',
                                                                  'WEATHER_FIRE_RISK_SCORE', 
                                                                  'WEATHER_FIRE_RISK_CATEGORY'
                                                                ]]

filtered_training_data_selectedColumn.to_csv('./0.Results_WeatherWildfireModel_PrepareDataSet/filtered_training_data_selectedColumn_Category_together.csv', index=False)

In [ ]:
m = create_map(filtered_training_data_selectedColumn, save_path='./0.Results_WeatherWildfireModel_PrepareDataSet/FireSpots.html', zoom_start=3)

In [ ]:
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
EXCLUDE_COLS = [
   'LATITUDE', 'LONGITUDE', 'NEAREST_WEATHER_LAT', 'NEAREST_WEATHER_LON', 'DISCOVERYDATETIME',  
   #'avg_max_temp_7d', 'avg_min_temp_7d', 'avg_rh_7d', 'total_precip_7d', 'max_wind_7d', 'consec_dry_days_7d', 'avg_solar_7d', 
   #'max_temp_7d', 'min_rh_7d', 'max_solar_7d',
   'avg_max_temp_30d', 'total_precip_30d', 'precip_anomaly_30d', 'avg_wind_30d', 'avg_rh_30d', 'days_no_rain_30d', 'avg_solar_30d', 
   'avg_max_temp_90d', 'max_temp_90d', 'total_precip_90d', 'days_no_rain_90d', 'avg_rh_90d', 'total_solar_90d', 
   #'days_above_heat_threshold', 'days_above_wind_threshold', 'days_low_rh', 
   #'IGNITION_WEATHER_SCORE', 'SEASONAL_SCORE', 'ASPECT_SCORE', 'ELEVATION_SCORE', 'SLOPE_WIND_SCORE', 'SPREAD_RATE_SCORE',
   #'WEATHER_FIRE_RISK_SCORE', 
   #'WEATHER_FIRE_RISK_CATEGORY'
]
# Note The commented out columns will remain in the dataframe

In [ ]:
plot_correlation_heatmap(filtered_training_data_selectedColumn, cols_to_exclude=EXCLUDE_COLS, show_numbers=False)